# Demo 15 — Comparing several factors with `posthoc_compare`

By default kbstatpy runs the pairwise post-hoc comparisons — and the data plot's significance brackets — on the **first** x-variable only. `options.posthoc_compare` lets you pick one or more factors to compare instead. Each listed factor is plotted as if it were the first variable (its levels on the x-axis, the others as facet panels), with its own significance brackets, and written to `DataPlots_<var>.*` and `Posthoc_<var>.xlsx`.

This demo reuses the two-way ToothGrowth model from Demo 3:

    len ~ supp * dose

but compares **both** factors in a single run, from one model fit — `supp` (orange juice vs ascorbic acid) and `dose` (low/medium/high). Comparisons are per-cell (conditional): each factor is compared within every cell of the other factor, so each facet panel gets its own brackets.

Other settings: `posthoc_compare = 'auto'` (the default) compares just the first factor; `posthoc_compare = 'none'` switches comparisons off (violins only, no brackets).

## Run on Google Colab

On [Google Colab](https://colab.research.google.com)? Run the cell below first —
it installs kbstatpy, its R packages, and the demo data (~3–5 min the first time).
It is a no-op when you run this notebook locally from the kbstatpy source tree.
Then run the cells below to see the tables and figures rendered inline.

In [ ]:
# Google Colab only: install kbstatpy + its R packages + the demo data.
# (Does nothing when the notebook runs locally from the source tree.)
import sys
if 'google.colab' in sys.modules:
    !curl -sSL https://raw.githubusercontent.com/kimbostroem/kbstatpy/master/demos/colab_setup.sh | bash

## Setup

In [ ]:
import os

import matplotlib.pyplot as plt
from kbstatpy import Kbstat, KbstatOptions

## Options

The same two-way model as Demo 3, with `posthoc_compare` set to compare both factors. Each comparison is plotted as if its factor were the first x-variable.

In [ ]:
options = KbstatOptions()
options.in_file         = os.path.join(options.demo_dir, 'data/toothgrowth.csv')
options.out_dir         = ''   # empty: show results inline only; set a folder to also save them
options.y               = 'len'
options.y_units         = 'mm'
options.x               = 'supp, dose'
options.interaction     = 'supp, dose'
options.posthoc_compare = 'supp, dose'   # compare BOTH factors, each as if first
options.rename          = 'len -> ToothLength; supp -> Supplement; dose -> Dose; supp: OJ -> orange_juice, VC -> vitamin_c'
options.x_order         = 'dose: low, medium, high'

## Run

`run()` executes the whole pipeline — fit, ANOVA, post-hoc comparisons, plots, and a printed summary — and writes files only if `out_dir` is set (empty here, so results are shown inline).

In [ ]:
kb = Kbstat(options)
kb.run();

## Save results

Everything above is shown inline. To also write all tables and figures to disk, uncomment the lines below and run this cell — it sets `out_dir` and calls `save()` to write the results already computed above (no re-run).

In [ ]:
# options.out_dir = 'results/demo_15_posthoc_compare'
# kb.save()
# kb.download_link()   # remote server: zip the results and click to download

## Interpretation

- Two data plots are produced: **`DataPlots_supp`** (Supplement on the x-axis, one panel per Dose) and **`DataPlots_dose`** (Dose on the x-axis, one panel per Supplement) — each with a matching `Posthoc_supp` / `Posthoc_dose` table.
- Comparisons are **per-cell**: each panel shows the pairwise tests of that factor *within that cell* of the other factor, so the brackets can differ between panels (e.g. medium-vs-high dose is *** under vitamin C but only * under orange juice). Each `Posthoc_<var>.xlsx` has the conditioning factor as a leading column.
- Reach for `posthoc_compare` whenever the factor you want to compare isn't the first one. Set it to `'none'` to drop the brackets entirely, or `'auto'` to compare only the first factor.
- Each `Posthoc_<var>.xlsx` also has a **marginal block** (every conditioning column = `any`) with the comparison averaged over the other factor — present in the table only, not the plot.